In [ ]:
import numpy as np
import pandas as pd

In [ ]:
dataset = pd.read_csv('datasets/kpi_with_congestion.csv')
dataset_cell_info = pd.read_csv('datasets/final_info_data.csv')

In [ ]:
dataset_cell_info.head()

In [ ]:
dataset_cell_info["Cell Name"].unique()

In [ ]:
extracted = (
            dataset_cell_info["Cell Name"]
            .astype(str)
            .str.strip()
            .str.extract(r"^([A-Za-z]{3})_(\d+)_([A-Za-z0-9]+)$")
        )
region = extracted[0].fillna(
    dataset_cell_info["Cell Name"].astype(str).str.extract(r"^([A-Za-z]{3})")[0]
).str.upper()
dataset_cell_info["Region"] = region
dataset_cell_info["Radio Site"] = region + "_" + extracted[1].fillna(dataset_cell_info["Cell Name"].astype(str).str.extract(r"^[A-Za-z]{3}_(\d+)")[0])


In [ ]:
dataset_cell_info["cell_id"] = dataset_cell_info["Cell Name"].str.extract(r'_([^_]+)$')

In [ ]:
dataset_cell_info["cell_id"].unique()

In [ ]:
dataset_cell_info["cell_id"].value_counts()

In [ ]:
import re

def get_frequency(cell_id):
    cell_id = str(cell_id).lower()
    # strip any leading digits (handles the 217l1 error case)
    cell_id = re.sub(r'^\d+', '', cell_id)
    # grab the first letter (ignores 'x' since it comes after, and digits)
    match = re.match(r'^([a-z])', cell_id)
    if not match:
        return 'Unknown'
    
    letter = match.group(1)
    
    frequency_map = {
        'h': '1800',
        'l': '2100',
        'f': '800',
        'j': '800',
        'g': 'TDD',
        'k': 'TDD',
    }
    return frequency_map.get(letter, 'Unknown')

dataset_cell_info['frequency'] = dataset_cell_info['cell_id'].apply(get_frequency)

In [ ]:
print(dataset_cell_info.groupby('frequency').size())
# or to verify the mapping is sensible:
print(dataset_cell_info.groupby(['frequency', 'cell_id']).size())

In [ ]:
dataset_cell_info.head()

In [ ]:
dataset_cell_info.info()

In [ ]:
dataset_cell_info.drop(columns=['cell_id'], inplace=True)

In [ ]:
dataset_cell_info.to_csv('datasets/final_info_data.csv', index=False)